# 2 — Extract a regional subset and bake the static website

Regenerates `../data/` for the GitHub Pages site. `../index.html` is the viewer and does not
change — only the data payload it fetches.

**Why range reads.** Each ARCO file is a global 0.25° field: 721 x 1440 x 24 int16 = ~50 MB for
one variable-day. A regional study needs a few tens of latitude rows out of 721. These are
NetCDF-3 (CDF-2) files, so each variable is one contiguous `int16` block at a known offset and
any latitude band is a single contiguous byte range per timestep. For Italy that is **3.3 MB per
file instead of 50 MB** — 15x less traffic, and it needs no Zarr store, no credentials and no
Icechunk.

The 2020 Italy extraction below ran 17,568 range requests with 0 failures and 0 missing values.

In [ ]:
import base64, gzip, io, json, os, struct, time, urllib.request
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BUCKET_HOST = "https://gcp-public-data-arco-era5.storage.googleapis.com"
BASE = BUCKET_HOST + "/raw/date-variable-single_level"
OUT  = Path("..")                      # repository root: index.html + data/

## A minimal NetCDF-3 reader

Only the header is parsed — enough to locate each variable's first byte, its shape and this
file's packing. Nothing else about the file is read.

In [ ]:
NCTYPE = {1: ("b", 1), 2: ("c", 1), 3: ("h", 2), 4: ("i", 4), 5: ("f", 4), 6: ("d", 8)}

def fetch(url, start=None, end=None, timeout=120, retries=4):
    headers = {} if start is None else {"Range": f"bytes={start}-{end}"}
    for attempt in range(retries):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=headers),
                                        timeout=timeout) as r:
                return r.read()
        except Exception:
            if attempt == retries - 1:
                raise
            time.sleep(1.5 * (attempt + 1))

def parse_nc3_header(buf):
    f = io.BytesIO(buf)
    assert f.read(3) == b"CDF"
    ver = f.read(1)[0]
    off_t, off_n = (">q", 8) if ver == 2 else (">i", 4)
    struct.unpack(">i", f.read(4))                       # numrecs
    def rd_str():
        n = struct.unpack(">i", f.read(4))[0]
        s = f.read(n).decode("utf-8", "replace"); f.read((-n) % 4); return s
    def rd_atts():
        struct.unpack(">i", f.read(4)); n = struct.unpack(">i", f.read(4))[0]
        atts = {}
        for _ in range(n):
            name = rd_str()
            nt, ne = struct.unpack(">i", f.read(4))[0], struct.unpack(">i", f.read(4))[0]
            fmt, sz = NCTYPE[nt]; raw = f.read(ne * sz); f.read((-(ne * sz)) % 4)
            atts[name] = raw.decode("utf-8", "replace") if nt == 2 else np.frombuffer(raw, ">" + fmt)
        return atts
    struct.unpack(">i", f.read(4)); nd = struct.unpack(">i", f.read(4))[0]
    dims = [(rd_str(), struct.unpack(">i", f.read(4))[0]) for _ in range(nd)]
    rd_atts()
    struct.unpack(">i", f.read(4)); nv = struct.unpack(">i", f.read(4))[0]
    variables = {}
    for _ in range(nv):
        name = rd_str(); ndims = struct.unpack(">i", f.read(4))[0]
        dimids = [struct.unpack(">i", f.read(4))[0] for _ in range(ndims)]
        atts = rd_atts(); struct.unpack(">i", f.read(4)); struct.unpack(">i", f.read(4))
        begin = struct.unpack(off_t, f.read(off_n))[0]
        variables[name] = {"shape": [dims[i][1] for i in dimids], "begin": begin, "atts": atts}
    return variables

## Domain

Latitude runs **north → south** from +90, longitude **eastward** from 0, both at 0.25°.
Change these four numbers to move the tool to another basin.

In [ ]:
LAT_N, LAT_S = 47.75, 36.00          # Italy
LON_W, LON_E =  6.00, 19.75
YEAR         = 2020
HOURLY_MONTH = 10                    # month kept at full hourly resolution

LAT0, LAT1 = int(round((90 - LAT_N) / 0.25)), int(round((90 - LAT_S) / 0.25)) + 1
LON0, LON1 = int(round(LON_W / 0.25)),        int(round(LON_E / 0.25)) + 1
NLAT, NLON, NGLON = LAT1 - LAT0, LON1 - LON0, 1440
lat = (90 - 0.25 * np.arange(LAT0, LAT1)).astype("float32")
lon = (0.25 * np.arange(LON0, LON1)).astype("float32")

dates = pd.date_range(f"{YEAR}-01-01", f"{YEAR}-12-31", freq="D")
VARS  = {"2m_temperature": "t2m", "total_precipitation": "tp"}
print(f"{NLAT} x {NLON} cells | {len(dates)} days | "
      f"{len(dates)*len(VARS)} files | ~{len(dates)*len(VARS)*NLAT*NGLON*2*24/1e9:.1f} GB transferred")

## Extract

One range request per timestep per file, all of them in one thread pool. Expect ~20 min for a full year of two variables.

In [ ]:
def day_url(d, var):
    return f"{BASE}/{d.year}/{d.month:02d}/{d.day:02d}/{var}/surface.nc"

HOURLY = {s: np.full((len(dates) * 24, NLAT, NLON), np.nan, np.float32) for s in VARS.values()}
tasks  = [(d, v) for d in dates for v in VARS]

def header_of(task):
    d, var = task
    url = day_url(d, var)
    return url, parse_nc3_header(fetch(url, 0, 16383))

t0 = time.time()
with ThreadPoolExecutor(24) as ex:
    headers = list(ex.map(header_of, tasks))
print(f"headers: {len(headers)} in {time.time()-t0:.0f}s")

jobs, ROWLEN = [], NGLON * 2
for (d, var), (url, H) in zip(tasks, headers):
    short = VARS[var]; V = H[short]
    sf = float(V["atts"]["scale_factor"][0]); ao = float(V["atts"]["add_offset"][0])
    fv = int(V["atts"]["_FillValue"][0]); di = (d - dates[0]).days
    for t in range(V["shape"][0]):
        jobs.append((url, V["begin"] + t * 721 * ROWLEN + LAT0 * ROWLEN,
                     sf, ao, fv, short, di * 24 + t))

failures = []
def run(job):
    url, off, sf, ao, fv, short, ti = job
    try:
        raw = np.frombuffer(fetch(url, off, off + NLAT * ROWLEN - 1),
                            dtype=">i2").reshape(NLAT, NGLON)[:, LON0:LON1]
        HOURLY[short][ti] = np.where(raw == fv, np.nan, raw.astype(np.float32) * sf + ao)
    except Exception as exc:
        failures.append((url, ti, str(exc)[:80]))

t0 = time.time()
with ThreadPoolExecutor(40) as ex:
    list(ex.map(run, jobs))
print(f"{len(jobs)} range reads in {time.time()-t0:.0f}s | failures {len(failures)}")
print("missing:", {s: float(np.isnan(a).mean()) for s, a in HOURLY.items()})

np.savez_compressed("era5_subset_hourly.npz", lat=lat, lon=lon, year=YEAR, **HOURLY)

## Physical units and temporal aggregation

`t2m` K → °C. `tp` is the accumulation over the hour **ending** at the stamp, in metres: m → mm,
and a daily total is the **sum** of 24 values, never a mean.

In [ ]:
nd = len(dates)
t2m_c  = HOURLY["t2m"] - 273.15
tp_mm  = np.clip(HOURLY["tp"] * 1000.0, 0, None)
daily_t  = t2m_c.reshape(nd, 24, NLAT, NLON).mean(1)
daily_tp = tp_mm.reshape(nd, 24, NLAT, NLON).sum(1)

h0 = (pd.Timestamp(YEAR, HOURLY_MONTH, 1) - dates[0]).days * 24
h1 = h0 + pd.Period(f"{YEAR}-{HOURLY_MONTH:02d}").days_in_month * 24

k = np.unravel_index(np.nanargmax(daily_tp), daily_tp.shape)
print(f"wettest cell-day: {daily_tp[k]:.1f} mm on {dates[k[0]].date()} "
      f"at {lat[k[1]]:.2f}N {lon[k[2]]:.2f}E")
print(f"domain annual total {daily_tp.sum(0).mean():.0f} mm | annual mean T {daily_t.mean():.2f} degC")

## Coastline

The 0.5 contour of the ERA5 land-sea mask on the same grid — the model's coast, hence blocky.

In [ ]:
H = parse_nc3_header(fetch(f"{BASE}/{YEAR}/01/01/land_sea_mask/surface.nc", 0, 16383))
V = H["lsm"]; off = V["begin"] + LAT0 * ROWLEN
lsm = (np.frombuffer(fetch(f"{BASE}/{YEAR}/01/01/land_sea_mask/surface.nc",
                           off, off + NLAT * ROWLEN - 1), dtype=">i2")
         .reshape(NLAT, NGLON)[:, LON0:LON1].astype(np.float32)
       * float(V["atts"]["scale_factor"][0]) + float(V["atts"]["add_offset"][0]))

fig, ax = plt.subplots()
coast = [[round(float(c), 3) for xy in p for c in xy]
         for p in ax.contour(lon, lat, lsm, levels=[0.5]).allsegs[0] if len(p) >= 2]
plt.close(fig)
print(f"{len(coast)} coastline segments, {sum(len(s)//2 for s in coast)} points")

## Quantise, compress, write

int16 with a fixed scale, byte-shuffled (all low bytes, then all high bytes — roughly 20 % better
gzip on smooth fields), gzipped. The steps below are far finer than ERA5's own accuracy:
0.05 °C, 0.005 mm h⁻¹, 0.05 mm d⁻¹.

In [ ]:
SCALES = {"t2m_h": 0.05, "tp_h": 0.005, "t2m_d": 0.05, "tp_d": 0.05}

def pack(a, scale):
    return np.clip(np.rint(a / scale), -32767, 32767).astype("<i2")

def shuffle_bytes(x):
    b = x.ravel().view(np.uint8).reshape(-1, 2)
    return np.concatenate([b[:, 0], b[:, 1]]).tobytes()

def write_array(a, scale, name):
    (OUT / "data" / name).write_bytes(gzip.compress(shuffle_bytes(pack(a, scale)), 9))
    return {"scale": scale, "offset": 0.0, "file": name}

(OUT / "data").mkdir(exist_ok=True)
manifest = {
    "lat": [round(float(v), 4) for v in lat],
    "lon": [round(float(v), 4) for v in lon],
    "coast": coast,
    "bases": {
        "hourly": {"start": f"{YEAR}-{HOURLY_MONTH:02d}-01T00:00:00Z", "step_h": 1,
                   "nt": h1 - h0,
                   "label": f"hourly - {pd.Timestamp(YEAR, HOURLY_MONTH, 1):%b %Y}",
                   "vars": {"tp":  write_array(tp_mm[h0:h1], SCALES["tp_h"],  "hourly_tp.bin.gz"),
                            "t2m": write_array(t2m_c[h0:h1], SCALES["t2m_h"], "hourly_t2m.bin.gz")}},
        "daily":  {"start": f"{YEAR}-01-01T00:00:00Z", "step_h": 24, "nt": nd,
                   "label": f"daily - {YEAR}",
                   "vars": {"tp":  write_array(daily_tp, SCALES["tp_d"],  "daily_tp.bin.gz"),
                            "t2m": write_array(daily_t,  SCALES["t2m_d"], "daily_t2m.bin.gz")}},
    },
    "presets": [{"name": "Whole domain",
                 "bbox": {"w": float(lon[0]), "s": float(lat[-1]),
                          "e": float(lon[-1]), "n": float(lat[0])}}],
    "meta": {"about": "<h3>Data</h3><p>ARCO-ERA5 raw single levels, extracted by HTTP range "
                      "reads of the regional latitude band.</p>"},
}
(OUT / "data" / "manifest.json").write_text(json.dumps(manifest, separators=(",", ":")),
                                            encoding="utf-8")
for f in sorted((OUT / "data").iterdir()):
    print(f"{f.name:22s} {f.stat().st_size/1e6:6.2f} MB")

## Verify the payload

Decode exactly the way the browser does and compare with the arrays in memory. The error should
be at most half a quantisation step.

In [ ]:
def browser_decode(entry, nt):
    u8 = np.frombuffer(gzip.decompress((OUT / "data" / entry["file"]).read_bytes()), np.uint8)
    n = len(u8) // 2
    v = (u8[:n].astype(np.int32) | (u8[n:].astype(np.int32) << 8))
    return (np.where(v > 32767, v - 65536, v) * entry["scale"]).reshape(nt, NLAT, NLON)

for label, entry, nt, ref in [
        ("hourly tp",  manifest["bases"]["hourly"]["vars"]["tp"],  h1 - h0, tp_mm[h0:h1]),
        ("daily  tp",  manifest["bases"]["daily"]["vars"]["tp"],   nd,      daily_tp),
        ("daily  t2m", manifest["bases"]["daily"]["vars"]["t2m"],  nd,      daily_t)]:
    print(f"{label}: max abs error {np.abs(browser_decode(entry, nt) - ref).max():.4g}")

## Deploy

```bash
git add -A && git commit -m "rebuild ERA5 payload" && git push
```

GitHub Pages serves the new `data/` within a minute or so. `index.html` does not need rebuilding —
it reads everything from `data/manifest.json` at load time.